# Phase Unwrapping Cloud Execution

This notebook sets up the `ali_proj` environment on Kaggle or Colab, clones the repository, installs dependencies via `uv`, and runs the training pipeline.

In [1]:
!nvidia-smi

Thu Feb 19 10:14:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

REPO_URL = "https://github.com/sattary/ali_proj.git"
BRANCH = "alis_code"
PROJECT_DIR = "ali_proj"

# 1. Clone Repository
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

# 2. Install uv
print("Installing uv...")
!pip instal uv

# 3. Sync Dependencies
print("Syncing dependencies...")
!cd {PROJECT_DIR} && uv sync

print("Setup complete!")

Cloning https://github.com/sattary/ali_proj.git (branch: alis_code)...
Cloning into 'ali_proj'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 260 (delta 112), reused 227 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (260/260), 347.54 KiB | 10.86 MiB/s, done.
Resolving deltas: 100% (112/112), done.
Installing uv...
ERROR: unknown command "instal" - maybe you meant "install"
Syncing dependencies...
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 68 packages in 1ms
Prepared 67 packages in 1m 21s                                           
Installed 67 packages in 716ms                              
 + alembic==1.18.4
 + ali-proj==0.1.0 (from file:///content/ali_proj)
 + annotated-doc==0.0.4
 + click==8.3.1
 + colorlog==6.10.1
 + contourpy==1.3.3
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.4
 + cycler==0.12.1
 +

In [6]:
import os
from getpass import getpass

# 1. Enter your credentials securely when prompted
# Paste your token (starting with ghp_) when the box appears
user = input("Enter GitHub Username: ")
email = input("Enter Email: ")
token = getpass("Paste GitHub Token: ")

# 2. Configure Git
!git config --global user.email {email}
!git config --global user.name {user}

In [3]:
!ls

ali_proj  sample_data


In [19]:
!cd {PROJECT_DIR} && ls -Fa && git status

./	  docs/       main.py	       README.md  src/	   .venv/
../	  .git/       pyproject.toml   results/   tests/
configs/  .gitignore  .python-version  scripts/   uv.lock
On branch alis_code
Your branch is up to date with 'origin/alis_code'.

nothing to commit, working tree clean


In [4]:
# Generate synthetic data
!cd {PROJECT_DIR} &&  uv run phase-unwrap generate --num-samples 20000 --out-dir data/cloud_test

Traceback (most recent call last):
  File "/content/ali_proj/.venv/bin/phase-unwrap", line 4, in <module>
    from phase_unwrap.cli import main
  File "/content/ali_proj/src/phase_unwrap/__init__.py", line 14, in <module>
    from .training.train import train
  File "/content/ali_proj/src/phase_unwrap/training/__init__.py", line 5, in <module>
    from .ablation import run_ablation
  File "/content/ali_proj/src/phase_unwrap/training/ablation.py", line 13, in <module>
    from .multiseed import run_multiseed
  File "/content/ali_proj/src/phase_unwrap/training/multiseed.py", line 16, in <module>
    from .train import CSV_COLUMNS, train
  File "/content/ali_proj/src/phase_unwrap/training/train.py", line 18, in <module>
    from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
  File "/content/ali_proj/.venv/lib/python3.12/site-packages/torchmetrics/__init__.py", line 37, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^

In [5]:
# Run training
!cd {PROJECT_DIR} && uv run phase-unwrap train --data-dir data/cloud_test --epochs 50 --device auto

Traceback (most recent call last):
  File "/content/ali_proj/.venv/bin/phase-unwrap", line 4, in <module>
    from phase_unwrap.cli import main
  File "/content/ali_proj/src/phase_unwrap/__init__.py", line 14, in <module>
    from .training.train import train
  File "/content/ali_proj/src/phase_unwrap/training/__init__.py", line 5, in <module>
    from .ablation import run_ablation
  File "/content/ali_proj/src/phase_unwrap/training/ablation.py", line 13, in <module>
    from .multiseed import run_multiseed
  File "/content/ali_proj/src/phase_unwrap/training/multiseed.py", line 16, in <module>
    from .train import CSV_COLUMNS, train
  File "/content/ali_proj/src/phase_unwrap/training/train.py", line 18, in <module>
    from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
  File "/content/ali_proj/.venv/lib/python3.12/site-packages/torchmetrics/__init__.py", line 37, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^

In [6]:
# visual check
!cd {PROJECT_DIR} && uv run phase-unwrap plot qualitative --checkpoint runs/default/latest.pth --data-dir data/cloud_test --out results/cloud_viz.png
from IPython.display import Image

Image(filename=f"{PROJECT_DIR}/results/cloud_viz.png")

Traceback (most recent call last):
  File "/content/ali_proj/.venv/bin/phase-unwrap", line 4, in <module>
    from phase_unwrap.cli import main
  File "/content/ali_proj/src/phase_unwrap/__init__.py", line 14, in <module>
    from .training.train import train
  File "/content/ali_proj/src/phase_unwrap/training/__init__.py", line 5, in <module>
    from .ablation import run_ablation
  File "/content/ali_proj/src/phase_unwrap/training/ablation.py", line 13, in <module>
    from .multiseed import run_multiseed
  File "/content/ali_proj/src/phase_unwrap/training/multiseed.py", line 16, in <module>
    from .train import CSV_COLUMNS, train
  File "/content/ali_proj/src/phase_unwrap/training/train.py", line 18, in <module>
    from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
  File "/content/ali_proj/.venv/lib/python3.12/site-packages/torchmetrics/__init__.py", line 37, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^

FileNotFoundError: [Errno 2] No such file or directory: 'ali_proj/results/cloud_viz.png'